In [1]:
import agents.encounter_designer as ed
import agents.enemy as e
import agents.npc as npc

import json
from pydantic import BaseModel, Field
from typing import List, Type, Optional

from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.chat_models import ChatOpenAI
from langchain.callbacks.manager import (
    AsyncCallbackManagerForToolRun,
    CallbackManagerForToolRun,
)
from langchain_core.agents import AgentFinish
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder
from langchain_core.tools import BaseTool, Tool

/home/vscode/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cura = npc.NPC(name="Captain Cura")
gually = npc.NPC(name="Gualluwyrm the Awesome")

/home/vscode/.local/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.chat_models.openai.ChatOpenAI` was deprecated in langchain-community 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [3]:
system_prompt = """
You are a D&D dungeon master and a powerful storyteller. You are a student of Joseph Campbell and you know how to structure the beats of an adventure.

You will design a story involving 2 to 4 thematic encounters.
The setting and party for the adventure is: {input}

You have the following tools:
EncounterDesigner - use this to design encounters appropriate for the players' levels, desired difficulty, and setting (In keeping with Joseph Campbell's insights, the difficulty should rise until we hit an exciting climax, but should never be too difficult for the players to overcome.)
StoryFormatter - use this to structure the story in the desired output format

Output Format: return an array of the encounters in the adventure. Use `StoryFormatter` to structure them into the desired format.
{agent_scratchpad}
"""

In [12]:
# TODO: Maybe we get rid of EncounterDesigner wholly and just use this? We can give it access to the sanity check tool to ensure the encounters are balanced
#
# Based on this comment from: https://hamel.dev/blog/posts/prompt/
# "I know from experience that you are likely to get better results if you tell the language model to generate ideas in one shot. That way, the LLM can reference previous ideas and achieve more diversity."
system_prompt = """
You are a D&D dungeon master and a powerful storyteller. You are a student of Joseph Campbell and you know how to structure the beats of an adventure.

You will design a story involving 2 to 4 thematic story hooks. In keeping with the teachings of Joseph Campbell, the drama should increase as the story continues.
The setting and party for the adventure is: {input}

You have the following tools:
StoryFormatter - use this to structure the story in the desired output format

Output Format: return an array of the story hooks in the adventure. Use `StoryFormatter` to structure them into the desired format.
{agent_scratchpad}
"""

In [5]:
system_prompt = """
You are a D&D dungeon master and a powerful storyteller. You are a student of Joseph Campbell and you know how to structure the beats of an adventure.

You will design a story involving 3 thematic story hooks. In keeping with Joseph Campbell's insights, the drama should increase as the story continues.
The setting and party for the adventure is: {input}

After you have come up with the story hooks, use them to generate encounters for the party to overcome.

You have the following tools:
EncounterDesigner - use this to design encounters appropriate for the players' levels, desired difficulty, setting, and story hook. IMPORTANT: You MUST include the desired difficulty and the story hook!
StoryFormatter - use this to structure the encounters in the desired output format

Output Format: return the encounters in the adventure. IMPORTANT: You MUST use `StoryFormatter` to structure them into the desired format.
{agent_scratchpad}
"""

In [13]:
class StoryHook(BaseModel):
    description: str = Field("description of the story hook or encounter")
    encounter: ed.Encounter = Field("the encounter for the story hook")

class StoryFormatterArgs(BaseModel):
    hooks: List[StoryHook] = Field("the events the players will experience throughout the story")

class StoryFormatter(BaseTool):
    name = "StoryFormatter"
    description = "format a set of encounters"
    args_schema: Type[BaseModel] = StoryFormatterArgs

    def _run(
        self, hooks: List[StoryHook],
        run_manager: Optional[CallbackManagerForToolRun] = None
    ):
        """Use the tool."""
        loggable = [s.model_dump() for s in hooks]
        return AgentFinish(return_values={"hooks": hooks}, log=json.dumps(loggable))
    
    async def _arun(
        self,
        hooks: List[str],
        run_manager: Optional[AsyncCallbackManagerForToolRun] = None,
    ) -> str:
        """Use the tool asynchronously."""
        raise NotImplementedError("does not support async")

In [14]:
tools = [
    StoryFormatter()
]

In [15]:
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo-1106")
prompt = ChatPromptTemplate.from_template(template=system_prompt)
agent = create_openai_tools_agent(llm, tools, prompt)

In [16]:
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
# f u langchain I can return direct if I want to
tools[-1].return_direct = True

In [18]:
executor.invoke({"input": "setting: a bustling bazaar, party: level 1 barbarian, level 1 ranger"})



> Entering new AgentExecutor chain...

Invoking: `StoryFormatter` with `{'hooks': [{'description': 'The Mysterious Merchant', 'encounter': {'enemies': [{'name': 'Mysterious Merchant', 'stats': {'armor_class': 12, 'hit_point': 15, 'speed': '30 ft.', 'initiative': '+1', 'strength': '10 (0)', 'dexterity': '14 (+2)', 'constitution': '12 (+1)', 'intelligence': '16 (+3)', 'wisdom': '14 (+2)', 'charisma': '18 (+4)'}, 'cr': '1/4', 'xp': 50, 'actions': [{'name': 'Enthralling Sales Pitch', 'description': 'The merchant attempts to enthrall the party with a sales pitch, causing them to make a Wisdom saving throw or be charmed for 1 minute.'}], 'other_info': 'The merchant is a charming and persuasive individual, with a knack for selling unusual and mysterious items.'}]}}, {'description': "The Thieves' Guild", 'encounter': {'enemies': [{'name': "Thieves' Guild Members", 'stats': {'armor_class': 13, 'hit_point': 10, 'speed': '30 ft.', 'initiative': '+2', 'strength': '11 (0)', 'dexterity': '16 (+3)'

{'input': 'setting: a bustling bazaar, party: level 1 barbarian, level 1 ranger',
 'output': "Here are the thematic story hooks for your adventure in the bustling bazaar:\n\n1. The Mysterious Merchant\n   - Encounter: Mysterious Merchant\n     - Description: The merchant is a charming and persuasive individual, with a knack for selling unusual and mysterious items.\n     - Enemies:\n       - Mysterious Merchant\n         - Armor Class: 12\n         - Hit Points: 15\n         - Speed: 30 ft.\n         - Initiative: +1\n         - Strength: 10 (0)\n         - Dexterity: 14 (+2)\n         - Constitution: 12 (+1)\n         - Intelligence: 16 (+3)\n         - Wisdom: 14 (+2)\n         - Charisma: 18 (+4)\n         - CR: 1/4\n         - XP: 50\n         - Actions:\n           - Enthralling Sales Pitch: The merchant attempts to enthrall the party with a sales pitch, causing them to make a Wisdom saving throw or be charmed for 1 minute.\n\n2. The Thieves' Guild\n   - Encounter: Thieves' Guild 

In [19]:
story = _

In [21]:
story["output"]

"Here are the thematic story hooks for your adventure in the bustling bazaar:\n\n1. The Mysterious Merchant\n   - Encounter: Mysterious Merchant\n     - Description: The merchant is a charming and persuasive individual, with a knack for selling unusual and mysterious items.\n     - Enemies:\n       - Mysterious Merchant\n         - Armor Class: 12\n         - Hit Points: 15\n         - Speed: 30 ft.\n         - Initiative: +1\n         - Strength: 10 (0)\n         - Dexterity: 14 (+2)\n         - Constitution: 12 (+1)\n         - Intelligence: 16 (+3)\n         - Wisdom: 14 (+2)\n         - Charisma: 18 (+4)\n         - CR: 1/4\n         - XP: 50\n         - Actions:\n           - Enthralling Sales Pitch: The merchant attempts to enthrall the party with a sales pitch, causing them to make a Wisdom saving throw or be charmed for 1 minute.\n\n2. The Thieves' Guild\n   - Encounter: Thieves' Guild Members\n     - Description: The thieves are skilled in stealth and thievery, and they are lo